In [1]:
import requests
from bs4 import BeautifulSoup
from datetime import datetime
import pandas as pd
from urllib.parse import urljoin, urlparse
import time # To add a delay between requests

def scrape_cancer_page(url):

    try:
        # Fetch the page content with a timeout
        response = requests.get(url, timeout=10)
        response.raise_for_status() # Raise an exception for HTTP errors
        soup = BeautifulSoup(response.text, 'html.parser')

        #Source Info
        source_organization = "Zorgwijzer.nl"
        source_page_title = soup.title.string.strip() if soup.title else "N/A"
        original_url = url
        scrape_date = datetime.now().strftime('%Y-%m-%d')

        #The actual content
        original_dutch_text = []
        hyperlinks = []

        # Attempt to find the main content block. Based on zorgwijzer.nl,
        main_content_div = soup.find('div', class_='entry-content')
        if not main_content_div:
            main_content_div = soup.find('article')
        if not main_content_div:
            main_content_div = soup.find('main', class_='main-content')

        if main_content_div:
            for p_tag in main_content_div.find_all('p'):
                text = p_tag.get_text(separator=' ', strip=True)
                if text:
                    original_dutch_text.append(text)

            # Extract hyperlinks within the main content div
            for a_tag in main_content_div.find_all('a', href=True):
                link_text = a_tag.get_text(strip=True)
                link_url = a_tag['href']
                # Ensuring the link text and URL are not empty and it's a valid link
                if link_text and link_url and not link_url.startswith('#'):
                    full_link_url = urljoin(url, link_url)
                    hyperlinks.append(f"{link_text} | {full_link_url}")
        else:
            for p_tag in soup.find_all('p'):
                text = p_tag.get_text(separator=' ', strip=True)
                if text:
                    original_dutch_text.append(text)
            for a_tag in soup.find_all('a', href=True):
                link_text = a_tag.get_text(strip=True)
                link_url = a_tag['href']
                if link_text and link_url and not link_url.startswith('#'):
                    full_link_url = urljoin(url, link_url)
                    hyperlinks.append(f"{link_text} | {full_link_url}")

        # Join all text pieces together
        full_dutch_text = '\n'.join(original_dutch_text)
        hyperlinks_str = '; '.join(hyperlinks)

        patient_journey_stage = ""
        care_pathway_topic = ""
        stakeholder = ""
        action_or_next_step = ""

        # Compile data
        data = {
            'Source organisation': source_organization,
            'Source page title': source_page_title,
            'Original URL': original_url,
            'Scrape date': scrape_date,
            'Original Dutch text': full_dutch_text,
            'Hyperlinks': hyperlinks_str,
            'Patient journey stage': patient_journey_stage,
            'Care pathway topic': care_pathway_topic,
            'Stakeholder': stakeholder,
            'Action or next step': action_or_next_step
        }
        return data

    except requests.exceptions.RequestException as e:
        print(f"Error fetching {url}: {e}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred while scraping {url}: {e}")
        return None

def find_internal_links(base_url, soup, domain):
    """
    Finds internal links on a given page that match a specific pattern.
    """
    internal_links = set()
    for a_tag in soup.find_all('a', href=True):
        href = a_tag['href']
        full_url = urljoin(base_url, href)
        parsed_url = urlparse(full_url)


        is_internal = parsed_url.netloc == domain
        is_relevant_path = parsed_url.path.startswith('/zorgwijzer/')
        is_not_anchor = '#' not in href

        is_likely_content = not any(kw in parsed_url.path for kw in ['/gebruiker/', '/contact/', '/over-ons/', '/disclaimer/', '/privacy/', '/algemene-voorwaarden/', '/vergoedingen/'])

        if is_internal and is_relevant_path and is_not_anchor and is_likely_content:
            clean_url = full_url.split('?')[0].split('#')[0]
            internal_links.add(clean_url)
    return internal_links


START_URL = "https://www.zorgwijzer.nl/zorgwijzer/kanker"
BASE_DOMAIN = urlparse(START_URL).netloc
MAX_PAGES_TO_SCRAPE = 20
SCRAPE_DELAY_SECONDS = 1

visited_urls = set()
urls_to_visit = {START_URL}
all_scraped_data = []
page_count = 0

print(f"Starting web scrape from: {START_URL}")
print(f"Max pages to scrape: {MAX_PAGES_TO_SCRAPE}")

while urls_to_visit and page_count < MAX_PAGES_TO_SCRAPE:
    current_url = urls_to_visit.pop()

    if current_url in visited_urls:
        continue

    print(f"Scraping page {page_count + 1}/{MAX_PAGES_TO_SCRAPE}: {current_url}")
    visited_urls.add(current_url)

    scraped_data = scrape_cancer_page(current_url)
    if scraped_data:
        all_scraped_data.append(scraped_data)
        page_count += 1


        try:
            response = requests.get(current_url, timeout=10)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, 'html.parser')
            new_links = find_internal_links(current_url, soup, BASE_DOMAIN)
            print(f"\tDiscovered {len(new_links)} new links on {current_url}:")
            for link in new_links:
                if link not in visited_urls and (page_count + len(urls_to_visit) < MAX_PAGES_TO_SCRAPE):
                    urls_to_visit.add(link)
                    print(f"\t\tAdding: {link}")
        except requests.exceptions.RequestException as e:
            print(f"Error fetching links from {current_url}: {e}")
        except Exception as e:
            print(f"An unexpected error occurred while finding links on {current_url}: {e}")

    time.sleep(SCRAPE_DELAY_SECONDS)

print("\n--- Scraping Complete --")

if all_scraped_data:
    df = pd.DataFrame(all_scraped_data)
    print(f"Scraped {len(df)} pages. Data preview:")
    display(df.head())

    # Export to CSV
    csv_filename = 'scraped_cancer_data.csv'
    df.to_csv(csv_filename, index=False)
    print(f"Data successfully exported to {csv_filename}")
else:
    print("No data was scraped.")


Starting web scrape from: https://www.zorgwijzer.nl/zorgwijzer/kanker
Max pages to scrape: 20
Scraping page 1/20: https://www.zorgwijzer.nl/zorgwijzer/kanker
	Discovered 0 new links on https://www.zorgwijzer.nl/zorgwijzer/kanker:

--- Scraping Complete --
Scraped 1 pages. Data preview:


,Source organisation,Source page title,Original URL,Scrape date,Original Dutch text,Hyperlinks,Patient journey stage,Care pathway topic,Stakeholder,Action or next step
0,Zorgwijzer.nl,Kanker: wat is het? (hulp & ondersteuning) - Z...,https://www.zorgwijzer.nl/zorgwijzer/kanker,2026-05-30,Kanker is een woord waar velen van zullen schr...,second opinion | https://www.zorgwijzer.nl/faq...,,,,


Data successfully exported to scraped_cancer_data.csv


In [2]:
csv_filename = 'scraped_cancer_data.csv'
df.to_csv(csv_filename, index=False)
print(f"Data successfully exported to {csv_filename}")

Data successfully exported to scraped_cancer_data.csv


In [4]:
"""
=============================================================================
 ZORGWIJZER.NL  —  Cancer Data Scraper
 Target  : https://www.zorgwijzer.nl/
 Output  : zorgwijzer_cancer_data.csv
 Language: Dutch pages scraped as-is; content columns ready for translation
=============================================================================

HOW TO RUN
----------
1. Install dependencies (once):
       pip install requests beautifulsoup4 pandas lxml

2. Run:
       python zorgwijzer_cancer_scraper.py

3. Output file:  zorgwijzer_cancer_data.csv  (in the same folder)

COLUMN REFERENCE
----------------
SOURCE INFO  (no translation needed)
  source_organisation   – Always "Zorgwijzer.nl"
  source_page_title_nl  – <title> tag, Dutch, kept as-is
  original_url          – Full URL of the scraped page
  scrape_date           – ISO date + time string when the page was captured
  content_hash          – SHA-256 fingerprint generated from the body text

CONTENT  (translate these columns later)
  original_dutch_text   – Main body text (paragraphs, headings, lists)
  hyperlinks            – JSON array of {"anchor": "...", "url": "..."} pairs
                          for every <a> tag found in the page body

TAGS  (left blank — your team fills these in later)
  patient_journey_stage     – e.g. Diagnosis / Treatment / Aftercare
  care_pathway_topic        – One of:
                                Referrals | Waiting Times | Treatment Decisions
                                Second Opinions | Additional Care | Communication
  stakeholder               – Patient | Caregiver | GP | Specialist | Other
  action_or_next_step       – Free text — what should the reader do next?
=============================================================================
"""

import re
import json
import time
import logging
import hashlib  # Added for generating the content hash
from datetime import datetime  # Upgraded from date to datetime for full timestamps
from urllib.parse import urljoin, urlparse

import requests
from bs4 import BeautifulSoup
import pandas as pd

# ─────────────────────────────────────────────────────────────
#  CONFIGURATION  — tweak these without touching the logic
# ─────────────────────────────────────────────────────────────

BASE_URL = "https://www.zorgwijzer.nl"

# Dutch keywords that indicate cancer-related content.
# The scraper keeps a page only if at least ONE of these appears
# (case-insensitive) anywhere in the page text or URL.
CANCER_KEYWORDS = [
    "kanker",          # cancer (generic)
    "oncologie",       # oncology
    "oncologisch",     # oncological
    "tumor",           # tumour
    "carcinoom",       # carcinoma
    "chemotherapie",   # chemotherapy
    "bestraling",      # radiation therapy
    "radiotherapie",   # radiotherapy
    "mammografie",     # mammography
    "borstkanker",     # breast cancer
    "longkanker",      # lung cancer
    "darmkanker",      # colon/bowel cancer
    "huidkanker",      # skin cancer
    "prostaatkanker",  # prostate cancer
    "leukemie",        # leukaemia
    "lymfoom",         # lymphoma
    "melanoom",        # melanoma
    "metastase",       # metastasis
    "palliatief",      # palliative
    "hospice",         # hospice
]

# Sitemap URLs to harvest page links from.
# Zorgwijzer publishes a standard sitemap_index; we also try the plain sitemap.
SITEMAP_URLS = [
    "https://www.zorgwijzer.nl/sitemap_index.xml",
    "https://www.zorgwijzer.nl/sitemap.xml",
    "https://www.zorgwijzer.nl/sitemap-0.xml",
]

# Seed pages to crawl when the sitemap yields nothing.
# These are known category/tag pages likely to link to cancer content.
SEED_PAGES = [
    "https://www.zorgwijzer.nl/zorgverzekering/kanker",
    "https://www.zorgwijzer.nl/faq/kanker-verzekering",
    "https://www.zorgwijzer.nl/zorgverzekering/oncologie",
    "https://www.zorgwijzer.nl/tag/kanker",
    "https://www.zorgwijzer.nl/ziektekosten/kanker",
    "https://www.zorgwijzer.nl/vergoeding/kanker",
    "https://www.zorgwijzer.nl/zorgverzekering/chemotherapie",
    "https://www.zorgwijzer.nl/zorgverzekering/bestraling",
    "https://www.zorgwijzer.nl/faq/oncologie",
]

# Crawl depth from seed pages (1 = only the seed; 2 = seed + linked pages, etc.)
MAX_CRAWL_DEPTH = 2

# Seconds to wait between HTTP requests — be polite to the server
REQUEST_DELAY = 1.5

# HTTP request timeout in seconds
REQUEST_TIMEOUT = 15

# Maximum pages to scrape in one run (safety cap — set to None for unlimited)
MAX_PAGES = 200

# Output file name
OUTPUT_FILE = "zorgwijzer_cancer_data.csv"

# ─────────────────────────────────────────────────────────────
#  LOGGING
# ─────────────────────────────────────────────────────────────

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# ─────────────────────────────────────────────────────────────
#  HTTP SESSION  — shared headers + session for keep-alive
# ─────────────────────────────────────────────────────────────

SESSION = requests.Session()
SESSION.headers.update({
    # Mimic a real browser so the server doesn't block the request
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "nl-NL,nl;q=0.9,en;q=0.8",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
})


# ─────────────────────────────────────────────────────────────
#  HELPER: safe GET request
# ─────────────────────────────────────────────────────────────

def safe_get(url: str) -> requests.Response | None:
    """
    Perform an HTTP GET for `url`.
    Returns the Response on success (2xx), None on any error.
    Always waits REQUEST_DELAY seconds before the actual call
    so we don't hammer the server.
    """
    time.sleep(REQUEST_DELAY)
    try:
        resp = SESSION.get(url, timeout=REQUEST_TIMEOUT, allow_redirects=True)
        if resp.status_code == 200:
            return resp
        log.warning("HTTP %s — %s", resp.status_code, url)
        return None
    except requests.RequestException as exc:
        log.error("Request failed for %s: %s", url, exc)
        return None


# ─────────────────────────────────────────────────────────────
#  STEP 1: Collect candidate URLs from the sitemap
# ─────────────────────────────────────────────────────────────

def fetch_sitemap_urls() -> list[str]:
    """
    Parse the site's XML sitemaps to get a flat list of page URLs.
    Handles both sitemap index files (which reference child sitemaps)
    and plain sitemap files (which list <loc> entries directly).
    Returns a deduplicated list of absolute URLs.
    """
    collected: set[str] = set()

    for sitemap_url in SITEMAP_URLS:
        log.info("Fetching sitemap: %s", sitemap_url)
        resp = safe_get(sitemap_url)
        if resp is None:
            continue

        soup = BeautifulSoup(resp.content, "xml")

        # Sitemap index → each <sitemap><loc> points to a child sitemap
        child_sitemaps = [loc.text.strip() for loc in soup.find_all("sitemap")]
        if child_sitemaps:
            log.info("  Found sitemap index with %d child sitemaps", len(child_sitemaps))
            for child_url in child_sitemaps:
                child_resp = safe_get(child_url)
                if child_resp is None:
                    continue
                child_soup = BeautifulSoup(child_resp.content, "xml")
                for loc in child_soup.find_all("loc"):
                    collected.add(loc.text.strip())
        else:
            # Plain sitemap — grab <loc> entries directly
            for loc in soup.find_all("loc"):
                collected.add(loc.text.strip())

    log.info("Sitemap harvest total: %d URLs", len(collected))
    return list(collected)


# ─────────────────────────────────────────────────────────────
#  STEP 2: Filter URLs to cancer-relevant ones
# ─────────────────────────────────────────────────────────────

def is_cancer_url(url: str) -> bool:
    """
    Quick pre-filter: return True if any CANCER_KEYWORD appears
    in the URL itself (e.g. /kanker, /oncologie).
    Full-text filtering happens after we fetch the page.
    """
    url_lower = url.lower()
    return any(kw in url_lower for kw in CANCER_KEYWORDS)


def is_cancer_page(text: str) -> bool:
    """
    Return True if the page body contains at least one cancer keyword.
    Used as the definitive gate before we add a row to the dataset.
    """
    text_lower = text.lower()
    return any(kw in text_lower for kw in CANCER_KEYWORDS)


# ─────────────────────────────────────────────────────────────
#  STEP 3: Crawl seed pages when sitemap yields few results
# ─────────────────────────────────────────────────────────────

def crawl_from_seeds(existing_urls: set[str], depth: int = MAX_CRAWL_DEPTH) -> set[str]:
    """
    BFS crawl starting from SEED_PAGES.
    At each depth level we fetch the page and collect <a href> links
    that belong to the same domain (zorgwijzer.nl).
    `existing_urls` prevents re-visiting pages already found in the sitemap.
    Returns a set of newly discovered internal URLs.
    """
    discovered: set[str] = set()
    frontier: set[str] = set(SEED_PAGES)

    for current_depth in range(depth):
        log.info("Crawl depth %d — frontier size: %d", current_depth + 1, len(frontier))
        next_frontier: set[str] = set()

        for url in frontier:
            if url in existing_urls or url in discovered:
                continue

            resp = safe_get(url)
            if resp is None:
                continue

            discovered.add(url)
            soup = BeautifulSoup(resp.content, "lxml")

            # Collect all internal links for the next depth level
            for a_tag in soup.find_all("a", href=True):
                href = a_tag["href"].strip()
                absolute = urljoin(BASE_URL, href)
                parsed = urlparse(absolute)

                # Keep only same-domain, HTTP(S) links — skip anchors, files, etc.
                if (
                    parsed.scheme in ("http", "https")
                    and "zorgwijzer.nl" in parsed.netloc
                    and not absolute.endswith((".pdf", ".jpg", ".png", ".zip"))
                    and "#" not in absolute
                ):
                    next_frontier.add(absolute.rstrip("/"))

        frontier = next_frontier - existing_urls - discovered

    log.info("Crawl discovered %d new URLs", len(discovered))
    return discovered


# ─────────────────────────────────────────────────────────────
#  STEP 4: Extract structured data from a single page
# ─────────────────────────────────────────────────────────────

def extract_main_content(soup: BeautifulSoup) -> str:
    """
    Pull the main readable text from the page.
    Priority order:
      1. <main> element
      2. <article> element
      3. A <div> with class containing 'content', 'article', or 'post'
      4. Fall back to <body>
    Strips script, style, nav, footer, and header noise.
    """
    # Remove noisy tags in place
    for noisy in soup(["script", "style", "nav", "footer", "header",
                        "aside", "form", "noscript", "svg", "iframe"]):
        noisy.decompose()

    # Try semantic content containers in priority order
    for selector in [
        "main",
        "article",
        "div.content",
        "div.article",
        "div.post",
        "div.entry-content",
        "div.page-content",
        "div#content",
        "div#main",
    ]:
        container = soup.select_one(selector)
        if container:
            return clean_text(container.get_text(separator="\n"))

    # Last resort — whole body
    body = soup.find("body")
    return clean_text(body.get_text(separator="\n")) if body else ""


def clean_text(raw: str) -> str:
    """
    Collapse excessive whitespace / blank lines from extracted text.
    Returns a readable, deduplicated multi-line string.
    """
    lines = [line.strip() for line in raw.splitlines()]
    # Drop empty lines and very short fragments (menu items, etc.)
    lines = [line for line in lines if len(line) > 2]
    # Collapse runs of blank lines to a single blank line
    cleaned = re.sub(r"\n{3,}", "\n\n", "\n".join(lines))
    return cleaned.strip()


def extract_hyperlinks(soup: BeautifulSoup, page_url: str) -> str:
    """
    Return a JSON string — array of {"anchor": "...", "url": "..."} objects —
    for every <a href> in the page body.
    Relative URLs are made absolute using the page's own URL.
    Example:
        [
          {"anchor": "Meer over kanker", "url": "https://www.zorgwijzer.nl/kanker"},
          ...
        ]
    """
    links = []
    seen_urls: set[str] = set()

    for a_tag in soup.find_all("a", href=True):
        href = a_tag["href"].strip()
        anchor_text = a_tag.get_text(strip=True)

        # Skip empty anchors and JavaScript pseudo-links
        if not href or href.startswith(("javascript:", "mailto:", "#")):
            continue

        absolute_url = urljoin(page_url, href)

        # Deduplicate by URL
        if absolute_url in seen_urls:
            continue
        seen_urls.add(absolute_url)

        links.append({"anchor": anchor_text or "(no text)", "url": absolute_url})

    # Return compact JSON — one object per link, no extra whitespace
    return json.dumps(links, ensure_ascii=False)


def scrape_page(url: str) -> dict | None:
    """
    Fetch `url`, parse it, and return a row-dict if the page is
    cancer-relevant.  Returns None if the page should be skipped.

    Fields returned:
      source_organisation, source_page_title_nl, original_url,
      scrape_date, content_hash, original_dutch_text, hyperlinks,
      patient_journey_stage, care_pathway_topic,
      stakeholder, action_or_next_step
    """
    resp = safe_get(url)
    if resp is None:
        return None

    soup = BeautifulSoup(resp.content, "lxml")

    # ── Page title (Dutch, kept as-is) ──────────────────────
    title_tag = soup.find("title")
    page_title = title_tag.get_text(strip=True) if title_tag else "(no title)"

    # ── Main body text ───────────────────────────────────────
    main_text = extract_main_content(soup)

    # ── Cancer relevance gate ────────────────────────────────
    # Combine URL, title, and body text for the keyword check
    combined = f"{url} {page_title} {main_text}".lower()
    if not is_cancer_page(combined):
        log.debug("Skipping (not cancer-related): %s", url)
        return None

    # ── Hyperlinks ───────────────────────────────────────────
    links_json = extract_hyperlinks(soup, url)

    # ── Content Hash & Timestamp Generation ──────────────────
    content_hash = hashlib.sha256(main_text.encode("utf-8")).hexdigest() if main_text else "N/A"
    full_timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # ── Assemble the row ─────────────────────────────────────
    return {
        # SOURCE INFO
        "source_organisation": "Zorgwijzer.nl",
        "source_page_title_nl": page_title,
        "original_url": url,
        "scrape_date": full_timestamp,       # Upgraded to full precise timestamp
        "content_hash": content_hash,        # Added unique cryptographic hash

        # CONTENT
        "original_dutch_text": main_text,
        "hyperlinks": links_json,

        # TAGS — left blank for the team to fill in
        "patient_journey_stage": "",       # e.g. Diagnosis / Treatment / Aftercare / Palliative
        "care_pathway_topic": "",          # Referrals | Waiting Times | Treatment Decisions |
                                           # Second Opinions | Additional Care | Communication
        "stakeholder": "",                 # Patient | Caregiver | GP | Specialist | Other
        "action_or_next_step": "",         # Free-text action item for the reader
    }


# ─────────────────────────────────────────────────────────────
#  STEP 5: Main orchestrator
# ─────────────────────────────────────────────────────────────

def main():
    log.info("=" * 60)
    log.info("Zorgwijzer.nl Cancer Scraper — starting")
    log.info("=" * 60)

    # ── Collect URLs from sitemap ────────────────────────────
    sitemap_urls = fetch_sitemap_urls()

    # Pre-filter to URLs that look cancer-related (fast, URL-only check)
    cancer_candidate_urls = {u for u in sitemap_urls if is_cancer_url(u)}
    log.info("Cancer-keyword URLs from sitemap: %d", len(cancer_candidate_urls))

    # ── Supplement with crawl from seed pages ────────────────
    crawled_urls = crawl_from_seeds(existing_urls=set(sitemap_urls))
    cancer_candidate_urls.update(crawled_urls)

    # Also add seed pages themselves if not already included
    for seed in SEED_PAGES:
        cancer_candidate_urls.add(seed)

    # ── De-duplicate and cap ─────────────────────────────────
    all_candidates = list(cancer_candidate_urls)
    if MAX_PAGES and len(all_candidates) > MAX_PAGES:
        log.info("Capping candidate list at %d (from %d)", MAX_PAGES, len(all_candidates))
        all_candidates = all_candidates[:MAX_PAGES]

    log.info("Total candidate pages to scrape: %d", len(all_candidates))

    # ── Scrape each page ─────────────────────────────────────
    rows: list[dict] = []
    for idx, url in enumerate(all_candidates, start=1):
        log.info("[%d/%d] Scraping: %s", idx, len(all_candidates), url)
        row = scrape_page(url)
        if row:
            rows.append(row)
            log.info("  ✓ Added row (total so far: %d)", len(rows))
        else:
            log.info("  — Skipped (not cancer-relevant or fetch error)")

    # ── Export to CSV ─────────────────────────────────────────
    if not rows:
        log.warning("No cancer-related rows collected. Check keywords or site accessibility.")
        return

    # Restructured column configuration to house new fields cleanly
    df = pd.DataFrame(rows, columns=[
        # SOURCE INFO
        "source_organisation",
        "source_page_title_nl",
        "original_url",
        "scrape_date",
        "content_hash",
        # CONTENT
        "original_dutch_text",
        "hyperlinks",
        # TAGS (blank — filled by team)
        "patient_journey_stage",
        "care_pathway_topic",
        "stakeholder",
        "action_or_next_step",
    ])

    # Ensure consistent column order even if some rows were missing keys
    df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
    # utf-8-sig adds a BOM so Excel opens the file correctly with Dutch characters

    log.info("=" * 60)
    log.info("Done! %d cancer pages exported → %s", len(rows), OUTPUT_FILE)
    log.info("=" * 60)

    # ── Quick summary ─────────────────────────────────────────
    print("\n📋 SCRAPE SUMMARY")
    print(f"   Pages collected : {len(rows)}")
    print(f"   Output file     : {OUTPUT_FILE}")
    print(f"   Scrape timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"\n   Columns ready for tagging:")
    print("     • patient_journey_stage")
    print("     • care_pathway_topic   (Referrals | Waiting Times | Treatment Decisions |")
    print("                             Second Opinions | Additional Care | Communication)")
    print("     • stakeholder          (Patient | Caregiver | GP | Specialist | Other)")
    print("     • action_or_next_step")
    print("\n   Columns ready for translation:")
    print("     • original_dutch_text")
    print("     • hyperlinks  (JSON array of anchor + URL pairs)\n")


# ─────────────────────────────────────────────────────────────
#  ENTRY POINT
# ─────────────────────────────────────────────────────────────

if __name__ == "__main__":
    main()

14:39:13  INFO      ============================================================
14:39:13  INFO      Zorgwijzer.nl Cancer Scraper — starting
14:39:13  INFO      ============================================================
14:39:13  INFO      Fetching sitemap: https://www.zorgwijzer.nl/sitemap_index.xml
14:39:15  INFO        Found sitemap index with 7 child sitemaps
14:39:17  WARNING   HTTP 404 — https://www.zorgwijzer.nl/post-sitemap.xml
2026-05-11T11:01:56+00:00
14:39:19  WARNING   HTTP 404 — https://www.zorgwijzer.nl/page-sitemap.xml
2026-04-14T09:55:05+00:00
14:39:20  WARNING   HTTP 404 — https://www.zorgwijzer.nl/zorgverzekering-sitemap.xml
2026-02-02T14:21:01+00:00
14:39:22  WARNING   HTTP 404 — https://www.zorgwijzer.nl/vergoeding-sitemap.xml
2026-02-23T09:05:36+00:00
14:39:24  WARNING   HTTP 404 — https://www.zorgwijzer.nl/faq-sitemap.xml
2026-04-15T12:09:24+00:00
14:39:25  WARNING   HTTP 404 — https://www.zorgwijzer.nl/category-sitemap.xml
2026-05-11T11:01:56+00:00
14:39:27  WA


📋 SCRAPE SUMMARY
   Pages collected : 6
   Output file     : zorgwijzer_cancer_data.csv
   Scrape timestamp: 2026-05-31 14:41:11

   Columns ready for tagging:
     • patient_journey_stage
     • care_pathway_topic   (Referrals | Waiting Times | Treatment Decisions |
                             Second Opinions | Additional Care | Communication)
     • stakeholder          (Patient | Caregiver | GP | Specialist | Other)
     • action_or_next_step

   Columns ready for translation:
     • original_dutch_text
     • hyperlinks  (JSON array of anchor + URL pairs)

